In [1]:
# !python --version
# import torch
# print(f"CUDA Available: {torch.cuda.is_available()}")
# print(f"GPU Name: {torch.cuda.get_device_name(0)}")  # Should show 'GeForce GTX 1050'

Python 3.11.3


ModuleNotFoundError: No module named 'torch'

In [1]:
!pip install torch torchvision torchaudio

In [ ]:
!pip install pytorch_tabular

In [2]:
import pandas as pd
import torch

from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

In [3]:
df = pd.read_csv('jan_data.csv', sep='\t', low_memory=False)

In [4]:
df.head()

,Unnamed: 0,Duration,Service,Source_bytes,Destination_bytes,Count,Same_srv_rate,Serror_rate,Srv_serror_rate,Dst_host_count,...,Label,Source_IP_Address,Source_Port_Number,Destination_IP_Address,Destination_Port_Number,Start_Time,Protocol,IDS_detection_alert_count,Malware_detection_alert_count,Ashula_detection_alert_count
0,0,1.501374,other,0,0,0,0.0,0.0,0.50,35,...,-1,fd75:41fb:cf76:fca4:019c:41cf:3af2:35c6,7681,fd75:41fb:cf76:39ef:7d8b:279c:615c:0d4d,445,00:00:00,tcp,0,0,0
1,1,2.589524,other,1621,324,1,1.0,0.0,0.67,1,...,1,fd75:41fb:cf76:36db:202e:034f:077c:556c,37273,fd75:41fb:cf76:dc4c:7d2c:2705:07b2:0f45,25,00:00:00,tcp,0,0,0
2,2,0.000000,other,0,0,0,0.0,0.0,0.50,23,...,-1,fd75:41fb:cf76:7cb9:4335:184e:535b:1e5a,2158,fd75:41fb:cf76:7f43:7d86:2789:6146:0399,445,00:00:01,tcp,0,0,0
3,3,10.855827,other,0,90,1,1.0,0.0,0.67,0,...,1,fd75:41fb:cf76:532a:0a93:ff00:07cd:2dbc,54567,fd75:41fb:cf76:dc4c:7d2c:2705:07b2:0f45,25,00:00:01,tcp,0,0,0
4,4,72.309494,other,100,0,0,0.0,0.0,0.50,0,...,-1,fd75:41fb:cf76:57c1:03dd:19c0:2010:7044,46992,fd75:41fb:cf76:b432:7d6b:276f:6080:3945,56674,00:00:01,udp,0,0,0


In [5]:
pd.set_option('display.max_columns', 200)

In [6]:
df = df.drop('Start_Time', axis=1)
df = df.drop('Unnamed: 0', axis=1)

In [7]:
df['Ashula_detection'] = df['Ashula_detection'].astype('str')
df['Label'] =df['Label'].astype('str')
df['Source_Port_Number'] = df['Source_Port_Number'].astype('str')
df['Destination_Port_Number'] = df['Destination_Port_Number'].astype('str')
df['Label'] = df['Label'].astype('str')

In [8]:
CAT_FEATURES = ['Service',
                'Flag',
                'IDS_detection',
                'Malware_detection',
                'Ashula_detection',
                'Source_IP_Address',
                'Source_Port_Number',
                'Destination_IP_Address',
                'Destination_Port_Number',
                'Protocol'
               ]
NUM_FEATURES = [
    'Destination_bytes',
    'Duration',
    'Dst_host_serror_rate',
    'Dst_host_srv_serror_rate',
    'Dst_host_srv_count',
    'Source_bytes',
    'Dst_host_count',
    'Serror_rate',
    'Count',
    'IDS_detection_alert_count',
    'Ashula_detection_alert_count',
    'Dst_host_same_src_port_rate',
    'Malware_detection_alert_count',
    'Srv_serror_rate',
    'Same_srv_rate'
]

In [9]:
from sklearn.model_selection import train_test_split

train_features = NUM_FEATURES + CAT_FEATURES
X = df[train_features]
y = df['Label']

In [10]:
X.shape

(3439069, 25)

In [11]:
y.shape

(3439069,)

In [12]:
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=32)

In [13]:
x_train.shape

(2579301, 25)

In [14]:
y_train.shape

(2579301,)

In [15]:
x_test.shape

(859768, 25)

In [ ]:
data_config = DataConfig(
    target=["Label"],
    continuous_cols=NUM_FEATURES,  # Numerical features
    categorical_cols=CAT_FEATURES,  # Categorical features
)

trainer_config = TrainerConfig(
    accelerator="gpu",
    devices=-1,
    precision=16,
    auto_lr_find=True,
    batch_size=32,
    max_epochs=1,
)

optimizer_config = OptimizerConfig()

model_config = TabTransformerConfig(
    task="classification",
    input_embed_dim=32,  # Embedding dimension
    num_heads=4,  # Number of attention heads
    num_attn_blocks=6,  # Number of transformer blocks
    learning_rate=1e-3,
)


tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

trainer = x_train.join(y_train)
tabular_model.fit(train=trainer)

In [ ]:
tabular_model.save_model("TabularIDSModel")

In [17]:
!pip install torchinfo

In [ ]:
print(tabular_model.ret_summary())

In [ ]:
preds = tabular_model.predict(x_test)

In [20]:
from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig

data_config = DataConfig(
    target=["Label"],
    continuous_cols=NUM_FEATURES,
    categorical_cols=CAT_FEATURES,
)

# Trainer config
trainer_config = TrainerConfig(
    accelerator="gpu",
    devices=-1,
    precision=16,
    auto_lr_find=True,
    batch_size=32,
    max_epochs=10,
)

# Optimizer config
optimizer_config = OptimizerConfig()

# FTTransformer model config
model_config = FTTransformerConfig(
    task="classification",
    learning_rate=1e-3,
    input_embed_dim=32,
    num_heads=4,
    num_attn_blocks=6,
    # transformer_dropout=0.1,
    # attention_dropout=0.1,
)

# Initialize TabularModel with FTTransformer
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

# Train model on your data
# x_train and y_train must be pandas DataFrames
trainer_data = x_train.join(y_train)
tabular_model.fit(train=trainer_data)

# Save model to folder
tabular_model.save_model("TabularIDSModel_FTT")

# Print model summary
print(tabular_model.ret_summary())

# Predict on test set
preds = tabular_model.predict(x_test)

# View predictions
print(preds.head())


2025-04-01 10:53:43,549 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


2025-04-01 10:53:53,854 - {pytorch_tabular.tabular_model:548} - INFO - Preparing the DataLoaders

2025-04-01 10:54:08,063 - {pytorch_tabular.tabular_datamodule:522} - INFO - Setting up the datamodule for          
classification task

2025-04-01 10:54:47,860 - {pytorch_tabular.tabular_model:599} - INFO - Preparing the Model: FTTransformerModel

...

Trainable params: 12.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 12.8 M                                                                                               
Total estimated model params size (MB): 51                                                                         
Modules in train mode: 142                                                                                         
Modules in eval mode: 0

Output()

2025-04-01 13:26:29,486 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-01 13:26:29,506 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

    | Name                                                                | Type                    | Params | Mode 
--------------------------------------------------------------------------------------------------------------------------
0   | _backbone                                                           | FTTransformerBackbone   | 173 K  | train
1   | _backbone.add_cls                                                   | AppendCLSToken          | 32     | train
2   | _backbone.transformer_blocks                                        | Sequential              | 172 K  | train
3   | _backbone.transformer_blocks.mha_block_0                            | TransformerEncoderBlock | 28.8 K | train
4   | _backbone.transformer_blocks.mha_block_0.mha                        | MultiHeadedAttention    | 16.4 K | train
5   | _backbone.transformer_blocks.mha_block_0.mha.to_qkv                 | Linear                  | 12.3 K | train
6   | _backbone.transformer_blocks.mha_block_0.mha.to_out 

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Assuming `y_test` contains the true labels for `x_test`
y_true = y_test.values.ravel()  # Convert to 1D array if needed
y_pred = preds["Label_prediction"].values  # Extract predicted labels

# Compute performance metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

# Print metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Display classification report
print("\nClassification Report:\n", classification_report(y_true, y_pred))

# Display confusion matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))


C:\Users\Rahul Siloniya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.6549
Precision: 0.4288
Recall: 0.6549
F1 Score: 0.5183


C:\Users\Rahul Siloniya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Rahul Siloniya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Rahul Siloniya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavi


Classification Report:
               precision    recall  f1-score   support

          -1       0.65      1.00      0.79    563025
          -2       0.00      0.00      0.00      4379
           1       0.00      0.00      0.00    292364

    accuracy                           0.65    859768
   macro avg       0.22      0.33      0.26    859768
weighted avg       0.43      0.65      0.52    859768


Confusion Matrix:
 [[563025      0      0]
 [  4379      0      0]
 [292364      0      0]]
